In [1]:
import travel_itinerary_agent.config  # loads .env and sets API key
from agents import Runner, trace
from travel_itinerary_agent.agents.intake import intake_agent
from travel_itinerary_agent.agents.visa import visa_agent
from travel_itinerary_agent.agents.activities import activities_agent

## 1. Intake: multi-turn chat


In [2]:
history = []

async def say(msg):
    global history
    result = await Runner.run(intake_agent, history + [{'role': 'user', 'content': msg}])
    history = result.to_input_list()
    out = result.final_output
    print('missing:', out.missing_fields)
    print('assistant:', out.next_question)
    return out

out = await say('I want to visit Japan with 2 friends next month for 7 days, budget 6000 USD')

missing: ['passport_country', 'origin', 'start_date', 'interests']
assistant: What country is your passport from?


In [3]:
out = await say('I hold a French passport, flying from Miami. Food and temples.')

missing: ['start_date']
assistant: What is your preferred start date for the trip?


In [4]:
trip = out.trip
print(trip.model_dump_json(indent=2))
print('complete:', out.is_complete)

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": null,
  "end_date": null,
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "Food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}
complete: False


## 2. Visa agent


In [5]:
with trace('Day4 visa'):
    r = await Runner.run(visa_agent, f'Passport: {trip.passport_country}. Destination: {trip.destination}.')
print(r.final_output.model_dump_json(indent=2))

{
  "passport": "France",
  "destination": "Japan",
  "requirement": "unknown",
  "allowed_days": null,
  "summary": "The visa requirement for French passport holders traveling to Japan is currently unverified. Please check the official government website of Japan for the most accurate and updated information.",
  "source": "mock"
}


## 3. Activities agent


In [6]:
prompt = (
    f'Destination: {trip.destination}. Dates: {trip.start_date} to {trip.end_date}. '
    f'Travelers: {trip.travelers}. Budget USD: {trip.budget_usd}. '
    f'Interests: {trip.interests}. Trip type: {trip.trip_type}.'
)
with trace('Day4 activities'):
    r = await Runner.run(activities_agent, prompt)
plan = r.final_output
print(plan.weather_summary)
for a in plan.activities:
    print('-', a.name, '|', a.category, '|', a.estimated_cost_usd)

In early June, Japan typically experiences warm temperatures with occasional rainfall, suitable for visiting outdoor temples and enjoying local cuisine.
- Kiyomizu-dera Temple | temple | 5.0
- Fushimi Inari Taisha | temple | 0.0
- Kinkaku-ji (Golden Pavilion) | temple | 4.0
- Senso-ji Temple | temple | 0.0
- Meiji Shrine | temple | 0.0
- Izakaya Food Tour in Tokyo | food | 80.0
- Tsukiji Outer Market | food | 30.0
- Kyoto Nishiki Market Food Walk | food | 40.0
